# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, we use its attributes
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we print the record set `@id`s, names, and field details, referencing entities by their `@id`.

In [ ]:
# List available record sets and their fields, referencing by @id
if not hasattr(metadata, 'record_sets'):
    # fallback in case .record_sets not provided
    print("No structured 'record_sets' found in the metadata.")
    record_set_ids = []
else:
    record_set_ids = []
    for recordset in metadata.record_sets:
        print(f"Record set @id: {getattr(recordset, '@id', None)} | name: {getattr(recordset, 'name', None)}")
        record_set_ids.append(getattr(recordset, '@id', None))
        if hasattr(recordset, 'fields'):
            for field in recordset.fields:
                print(f"    Field @id: {getattr(field, '@id', None)} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")
        elif hasattr(recordset, 'columns'):  # fallback
            for col in recordset.columns:
                print(f"    Column @id: {getattr(col, '@id', None)} | name: {getattr(col, 'name', None)} | dataType: {getattr(col, 'data_type', None)}")

# If the metadata provides no record_sets, attempt to probe what is available (for exploratory purpose only)
if not record_set_ids:
    try:
        # Try to print available datasets
        print("Exploring record sets from dataset.records.")
        # Try to find from dataset._schema in edge cases
        if hasattr(dataset, '_schema'):
            import json
            rs = []
            for rec in dataset._schema.get('recordSet', []):
                rsid = rec.get('@id', None)
                print(f"RecordSet @id: {rsid} | name: {rec.get('name', None)}")
                rs.append(rsid)
                if 'field' in rec:
                    for field in rec['field']:
                        print(f"    Field @id: {field.get('@id', None)} | name: {field.get('name', None)} | dataType: {field.get('dataType', None)}")
                elif 'column' in rec:
                    for col in rec['column']:
                        print(f"    Column @id: {col.get('@id', None)} | name: {col.get('name', None)} | dataType: {col.get('dataType', None)}")
            record_set_ids = rs
    except Exception as e:
        print("Could not extract record sets: ", e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.


In [ ]:
# Example: List records for each found record set (by @id)
import pprint
dataframes = {}

# Use the discovered record_set_ids; if none found above, attempt default
if not record_set_ids or record_set_ids == [None]:
    # Manually specify likely record set id (see Croissant schema if necessary)
    # For this dataset, try '@id' in form 'cr:protein_record_set' (example)
    print("No record_set @id detected, attempting with default 'cr:ProteinRecordSet'.")
    record_set_ids = ['cr:ProteinRecordSet']

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"RecordSet {record_set_id} columns:\n{df.columns.tolist()}")
            display(df.head())  # Show sample
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Could not extract records for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data, or grouping by attributes. 

Below, we demonstrate filtering, normalization, and grouping using field `@id`s.

In [ ]:
# Pick one of the loaded record sets
# Replace these IDs after you run the previous overview to match your schema
record_set_id = next(iter(dataframes)) if dataframes else None

if record_set_id is not None:
    df = dataframes[record_set_id]
    # Try to automatically select a numeric field (replace with your numeric field @id)
    numeric_field = None
    for col in df.columns:
        # Guess numeric columns by checking dtype or name
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to guess a field name (e.g., coverage, MW, abundance)
        for col in df.columns:
            if any(x in col.lower() for x in ['coverage', 'abundance', 'peptide', 'count', 'mw', 'mass']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if pd.api.types.is_numeric_dtype(df[col]):
                        numeric_field = col
                        break
                except Exception:
                    pass
    print(f"Using numeric field '@id': {numeric_field}")

    # Set a filter threshold (example: 10)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (count={len(filtered_df)}):")
    display(filtered_df.head(3))

    # Normalize the numeric field (z-score)
    filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' (first 3 rows):")
    display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head(3))

    # Try to find a grouping field (e.g., based on string columns)
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < len(df)//2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped filtered data by '{group_field}' (showing mean '{numeric_field}'): ")
        print(grouped_df.head())
else:
    print("No valid DataFrame extracted for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.show()

    if group_field:
        plt.figure(figsize=(12, 4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (filtered)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and inspect the FAIR\u2072 mass spectrometry dataset. We listed record sets and their fields by `@id`, loaded data into DataFrames, filtered and normalized a numeric field, grouped data by a categorical attribute, and visualized key distributions. Referencing dataset elements by `@id` ensures reproducibility and schema compliance. You can further analyze the dataset by selecting other record sets or fields as relevant to your research.